## 4 — GLMERStan (state-level estimates, CES sample2 / 3000)
Bayesian Multilevel Regression + Poststratification via `rstanarm` (R) called from Python using `rpy2`.  
Outcomes: `climate_problem` and `renewable_fuel`

**Howe (2015) 3-level architecture** estimated via full MCMC (NUTS/HMC):
- Individual: `logit(p_i) = γ₀ + α_gender + α_race + α_educ + α_state`
- State: `α_state[s] ~ N(α_region[div[s]] + γ_carbon·co2_std + γ_pres·pres_std + γ_drive·drive_std + γ_ss·samesex_std, σ_state²)`
- Region: `α_region[r] ~ N(0, σ_region²)` — 9 Census divisions

**Requires:** R ≥ 4.0, `rstanarm` R package, `rpy2` Python package.
before running:
##conda activate deepverse
##pip install -r requirements.txt

In [1]:
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

import rpy2.robjects as ro
from rpy2.robjects import pandas2ri

sys.path.insert(0, str(Path('.').resolve()))
from utils import (
    OUTPUT_DIR, STATE_FIPS_TO_NAME,
    SURVEY_PATH, POSTSTRAT_STATE_PATH,
    _load_state_covariates, save_estimates,
)

DATA_DIR   = Path("../../")
OUTCOME    = ['climate_problem', 'renewable_fuel']
MODEL_NAME = 'glmerstan'
SEED       = 42

In [2]:
%load_ext rpy2.ipython

### 1. Load and recode data

In [3]:
raw = pd.read_csv(SURVEY_PATH, dtype={'state_fips': str})

ps_frame = pd.read_csv(POSTSTRAT_STATE_PATH, dtype={'state_fips': str})

state_cov = _load_state_covariates()

def _std(x):
    return (x - x.mean()) / x.std()

state_cov['co2_std']     = _std(state_cov['co2_per_capita'])
state_cov['pres_std']    = _std(state_cov['dem_share_two_party'])
state_cov['drive_std']   = _std(state_cov['drive_alone_share'])
state_cov['samesex_std'] = _std(state_cov['samesex_share'])

# attach division (numeric 1–9) from poststrat frame
state_div = ps_frame.groupby('state_fips')['division'].first().reset_index()
state_cov = state_cov.merge(state_div, on='state_fips', how='left')

cov_std = ['state_fips', 'co2_std', 'pres_std', 'drive_std', 'samesex_std', 'division']

# ps_frame already has 'division'; drop it before merging to avoid division_x/division_y collision
ps_r = ps_frame.drop(columns=['division']).merge(state_cov[cov_std], on='state_fips', how='left')
ps_r['division']       = ps_r['division'].astype(str)
ps_r['educ_category']  = ps_r['educ_category'].astype(str)

print(f'Survey raw: {raw.shape}  |  Poststrat: {ps_frame.shape} | {ps_frame["state_fips"].nunique()} states')


Survey raw: (3000, 23)  |  Poststrat: (1632, 9) | 51 states


### 2–4. Fit + poststratify for each outcome

In [4]:
_cv = ro.default_converter + pandas2ri.converter

for OUTCOME_VAR in OUTCOME:
    print(f'\n{"="*60}\nOutcome: {OUTCOME_VAR}\n{"="*60}')

    survey = raw.dropna(subset=['gender', 'educ_category', OUTCOME_VAR]).copy()
    survey[OUTCOME_VAR]     = survey[OUTCOME_VAR].astype(int)
    survey['educ_category'] = survey['educ_category'].astype(str)

    survey_r = survey.merge(state_cov[cov_std], on='state_fips', how='left')
    survey_r['division'] = survey_r['division'].astype(str)

    print(f'Respondents: {len(survey_r):,}  ({survey_r[OUTCOME_VAR].mean()*100:.1f}% support)')

    with _cv.context():
        ro.globalenv['survey_r'] = survey_r
    ro.globalenv['outcome'] = OUTCOME_VAR

    ro.r(f'''
    suppressPackageStartupMessages(library(rstanarm))

    survey_r$gender        <- as.factor(survey_r$gender)
    survey_r$race4         <- as.factor(survey_r$race4)
    survey_r$educ_category <- as.factor(survey_r$educ_category)
    survey_r$state_fips    <- as.factor(survey_r$state_fips)
    survey_r$division      <- as.factor(survey_r$division)

    form <- as.formula(paste0(
        "{OUTCOME_VAR} ~ co2_std + pres_std + drive_std + samesex_std +",
        "(1 | division) + (1 | state_fips) +",
        "(1 | gender) + (1 | race4) + (1 | educ_category)"
    ))

    fit <- stan_glmer(
        form,
        data             = survey_r,
        family           = binomial(link = 'logit'),
        prior            = normal(0, 1, autoscale = FALSE),
        prior_intercept  = normal(0, 1.5, autoscale = FALSE),
        prior_covariance = decov(regularization=1, concentration=1, shape=1, scale=2.5),
        chains           = 4L,
        iter             = 3000L,
        warmup           = 2000L,
        seed             = 42L,
        cores            = 4L,
        adapt_delta      = 0.95,
        refresh          = 0
    )

    cat('\\n=== Fixed Effects ===\\n')
    print(round(fixef(fit), 4))
    cat('\\n=== Random Effect SDs ===\\n')
    print(VarCorr(fit))
    rhat_vals <- summary(fit)[, 'Rhat']
    cat(sprintf('Max Rhat: %.3f\\n', max(rhat_vals, na.rm=TRUE)))
    ''')

    with _cv.context():
        ro.globalenv['ps_r'] = ps_r
    ro.r('''
    ps_r$gender        <- as.factor(ps_r$gender)
    ps_r$race4         <- as.factor(ps_r$race4)
    ps_r$educ_category <- as.factor(ps_r$educ_category)
    ps_r$state_fips    <- as.factor(ps_r$state_fips)
    ps_r$division      <- as.factor(ps_r$division)

    draws        <- posterior_epred(fit, newdata=ps_r, allow_new_levels=TRUE)
    cell_preds_r <- data.frame(
        state_fips     = as.character(ps_r$state_fips),
        N              = ps_r$N,
        predicted_prob = colMeans(draws)
    )
    ''')

    with _cv.context():
        cell_preds = ro.conversion.get_conversion().rpy2py(ro.globalenv['cell_preds_r'])

    result = (
        cell_preds
        .groupby('state_fips')
        .apply(lambda g: np.average(g['predicted_prob'], weights=g['N']), include_groups=False)
        .reset_index(name='estimate')
    )
    result['state_name'] = result['state_fips'].map(STATE_FIPS_TO_NAME)

    save_estimates(result, MODEL_NAME, OUTCOME_VAR)

    print(result.sort_values('estimate', ascending=False).head(10).to_string(index=False))


Outcome: climate_problem
Respondents: 2,977  (62.8% support)



=== Fixed Effects ===


(Intercept) 

    co2_std 

   pres_std 

...

R[write to console]: 1: There were 136 divergent transitions after warmup. See
https://mc-stan.org/misc/warnings.html#divergent-transitions-after-warmup
to find out why this is a problem and how to eliminate them. 



R[write to console]: 2: Examine the pairs() plot to diagnose sampling problems
 



R[write to console]: 3: Bulk Effective Samples Size (ESS) is too low, indicating posterior means and medians may be unreliable.
Running the chains for more iterations may help. See
https://mc-stan.org/misc/warnings.html#bulk-ess 



R[write to console]: 4: Tail Effective Samples Size (ESS) is too low, indicating posterior variances and tail quantiles may be unreliable.
Running the chains for more iterations may help. See
https://mc-stan.org/misc/warnings.html#tail-ess 




  glmerstan (renewable_fuel) — State-Level Estimates
  States with estimates: 51
  States with NaN:       0
  Mean estimate:         0.5762
  Median estimate:       0.5777
  Min estimate:          0.3268
  Max estimate:          0.7622

  Saved → /Users/kaeleyoshea/Capstone/A.MRdeeP-Deep-Learning--MRP/model_run_ces/sample2_state/outputs/estimates/renewable_fuel_state_estimates.csv

state_fips  estimate           state_name
        11  0.762209 District of Columbia
        24  0.693959             Maryland
        36  0.689464             New York
        34  0.667005           New Jersey
        06  0.663381           California
        15  0.658846               Hawaii
        17  0.648858             Illinois
        25  0.648758        Massachusetts
        09  0.640236          Connecticut
        51  0.632811             Virginia
